Сравнить между собой нижние оценки для задачи коммивояжера $(\geq 4)$ на примере **tsp.txt**.

In [2]:
import numpy as np

def read_tsp(filename):

    coordinates = {}
    dimension = None
    edge_weight_type = None
    reading_coordinates = False

    with open(filename, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            if line == "EOF":
                break

            if line == "NODE_COORD_SECTION":
                reading_coordinates = True
                continue

            if not reading_coordinates:
                # Заголовки TSPLIB
                if ":" in line:
                    key, value = line.split(":", 1)
                    key = key.strip()
                    value = value.strip()

                    if key == "DIMENSION":
                        dimension = int(value)

                    elif key == "EDGE_WEIGHT_TYPE":
                        edge_weight_type = value

            else:
                parts = line.split()

                city_id = int(parts[0])
                x = float(parts[1])
                y = float(parts[2])

                if city_id in coordinates:
                    raise ValueError(f"Повторный номер города: {city_id}")

                coordinates[city_id] = (x, y)

    city_ids = sorted(coordinates)

    coordinates = np.array(
        [coordinates[i] for i in city_ids],
        dtype=float
    )

    dx = coordinates[:, 0, None] - coordinates[None, :, 0]
    dy = coordinates[:, 1, None] - coordinates[None, :, 1]

    C = np.floor(np.sqrt(dx**2 + dy**2) + 0.5).astype(int)

    np.fill_diagonal(C, 0)

    return coordinates, C


coordinates, C = read_tsp("tsp.txt")

n = len(coordinates)

print("Количество городов:", n)
print("Размер матрицы расстояний:", C.shape)
print("Первые 5 координат:")
print(coordinates[:5])

print("\nФрагмент матрицы расстояний:")
print(C[:5, :5])

C_work = C.astype(float).copy()
np.fill_diagonal(C_work, np.inf)

np.save("distance_matrix.npy", C)
np.save("coordinates.npy", coordinates)


Количество городов: 395
Размер матрицы расстояний: (395, 395)
Первые 5 координат:
[[ 0. 15.]
 [ 0. 22.]
 [ 0. 29.]
 [ 0. 42.]
 [ 0. 43.]]

Фрагмент матрицы расстояний:
[[ 0  7 14 27 28]
 [ 7  0  7 20 21]
 [14  7  0 13 14]
 [27 20 13  0  1]
 [28 21 14  1  0]]


Расстояние между горадами вычисляем как: $$c_{ij} = \left\lfloor\sqrt{(x_i - x_j)^2 + (y_i - y_j)^2} + 0.5 \right\rfloor$$

Нижняя оценка - это величина, которая не превосходит оптимального значения целевой функции в задаче минимизации. Для задачи коммивояжера: $$LB \leq OPT$$

Где:   
  
$LB$ - нижняя оценка длины оптимального маршрута  
$OPT$ - длина кратчайшего гамильтонова цикла  
$c_{ij}$ - стоимость перехода из города $i$ в город $j$  
$n$ - количество городов

### Нижняя оценка через минимальное остовное дерево.
Дан связный граф $$G = (V,E)$$
Каждому ребру $e \in E$ приписан вес $w_{e} \geq 0$
Требуется найти в графе остовное дерево с минимальным суммарным весом ребер.

По алгоритму **Крускала**:
1. Упорядочиваем рёбра графа в порядке неубывания весов.  
2. Начинаем с графа без рёбер.  
3. Последовательно добавляем рёбра, если их добавление не приводит к образованию цикла.  
4. Продолжаем, пока не будет добавлено $n-1$ ребер.

Полученное дерево явялется минимальным остовным деревом.  

Тогда имеем теорему: 
Алгоритм Крускала дает нижнюю оценку для задачи коммивояжера:  
$$A_{k}(I) \leq OPT(I)$$

### Примитивная нижняя оценка

Примитивная нижняя оценка строится через две величины: плату за въезд и плату за выезд.  

Плата за выезд из города $i$: $$a_i = \min_{i \neq j}c_{ij}$$

Плата за въезд в город $j$: $$b_j = \min_{i \neq j}(c_{ij} - a_i)$$

Тогда $$OPT(c_{ij}) \geq \sum_{i=1}^{n}a_i + \sum_{j=1}^{n}b_{j}$$

Таким образом, примитивная нижняя оценка имеет вид:

$$LB_{прим} = \sum_{i=1}^{n}a_i + \sum_{j=1}^{b}b_j$$


### Нижняя оценка линейного программирования
Введем переменные $$x_{ij} = \begin{cases} 1, \ если \ из \ города \ i \ едем \ в \ город \ j \\ 0, \ в \ противном \ случае \end{cases}$$

Целевая функция: $$\sum_{i=1}^{n}\sum_{j=1}^{n}c_{ij} \cdot x_{ij} \rightarrow \min$$

Ограничения:
(Въезжаем в каждый город ровно один раз)
$$\sum_{i=1}^{n}x_{ij} = 1, \ j \in J$$
(Из каждого города выезжаем ровно один раз)
$$\sum_{j=1}^{n}x_{ij} = 1, \ i \in J$$
(Исключение подциклов)
$$\sum_{i \in S}\sum_{j \in J/S}x_{ij} \geq 1, \ \forall S \subseteq J, \ S \neq \emptyset$$
(Условие целочисленности)
$$x_{ij} \in \{0,1\}, \ i,j \in J$$

Получаем нижнюю оценку через замену $x_{ij} \in \{0,1\} \rightarrow 0 \leq x_{ij} \leq 1$

Получаем задачу линейного программирования, оптимальное значение которой является нижней оценкой: $$LB_{ЛП} \leq OPT$$

### Нижняя оценка через 1-деревья
